# **Confidence Intervals and Statistical Comparison of XAI Methods**

In [1]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from IPython.display import Markdown, display
import pandas as pd

In [2]:
DATA_DIR = Path("/data/aveera/results") #Path to where all robustness and faithfulness results have been saved
OUTPUT_DIR = Path("/data/aveera/results/ci_results") # Path to where confidence interval results will be saved

IROF_FILES = ["irof_full_efficientnetb0_test_2026-10-18.csv","irof_full_resnet50_test_2026-10-18.csv",] # Names of IROF files
RISE_FILE = "rise_original_full.csv" # Name of Rise files 
MAXSENS_FILE="maxsens_checkpoints/maxsens_per_image_checkpoint.csv" # Name of max sensitivity files

N_BOOT = 10000
ALPHA = 0.05
SEED = 42

MODELS = ["resnet50","efficientnetb0"]
METHODS = ["GradCAM", "LIME", "SHAP"]

LOWER_IS_BETTER = {
    "irof_auc":True,
    "del_auc":True,
    "ins_auc":False,
    "max_sensitivity":True,
    "ssim_dist":True,
}

KEYS = ["model","method","image_index"]

In [3]:
def load_long() -> pd.DataFrame:
    """Load all result files and combine them into one long-format table with one row per (model, method, image, metric). """
    irof=pd.concat([pd.read_csv(DATA_DIR/f) for f in IROF_FILES])[KEYS+["irof_auc"]]
    rise=pd.read_csv(DATA_DIR/RISE_FILE)[KEYS+["del_auc","ins_auc"]]
    ms=pd.read_csv(DATA_DIR/MAXSENS_FILE)[KEYS+ ["max_sensitivity","ssim_dist"]]

    wide = irof.merge(rise, on=KEYS, how="outer").merge(ms, on=KEYS, how="outer")
    wide = wide[wide["method"].isin(METHODS+["Random"])]
    long = wide.melt(id_vars=KEYS, value_vars=list(LOWER_IS_BETTER),var_name="metric",value_name="value").dropna(subset=["value"],how="all")

    return long

def boot_ci(x,rand):
    """Mean and percentile bootstrap CI. NaNs are dropped"""
    x=np.asarray(x,dtype=float)
    x=x[np.isfinite(x)] # Remove NaN and infinite values
    n=len(x)
    idx=rand.integers(0,n,size=(N_BOOT,n)) # Draw 10 000 resamples of the same size as the data
    means= x[idx].mean(axis=1)
    low, high = np.percentile(means, [100 * ALPHA / 2, 100*(1-ALPHA/2)])
    return x.mean(), low, high, n

def paired_test(a,b,rand):
    """Compare two sets of per-image scores (a and b) that are paired by image. Returns the mean difference witha  bootstrap CI and a Wilcoxon p-value"""
    d=(a-b).dropna() # per image difference
    mean_d, low, high, n =boot_ci(d.values,rand) # Bootstrap CI on the mean difference
    try:
        p=stats.wilcoxon(d.values).pvalue
    except ValueError:
        p=1.0
    return {"n_pairs":n, "mean_diff":mean_d,"ci_low":low,"ci_high":high,"median_diff":d.median(),"p_raw":p}

def add_holm_and_winner(df, family_col):
    """Apply Holm correction within each family of tests, flag significance and decide which side (A or B) performed better"""
    df = df.copy()
    df["p_holm"]=np.nan

    # Correct p-values seperately within each family
    for _,grp in df.groupby(family_col):
        df.loc[grp.index,"p_holm"] = multipletests(grp["p_raw"],method="holm")[1]
    df["ci_excludes_0"]=(df["ci_low"]>0) | (df["ci_high"] <0 )
    df["significant_holm"]=df["p_holm"] < ALPHA
    df["evidence"] = np.where(df["significant_holm"] & df["ci_excludes_0"], "both",np.where(df["significant_holm"] | df["ci_excludes_0"], "mixed","none"))

    def winner(r):
        # Only declare a winner if the Holm-corrected test is significant
        if not r["significant_holm"]:
            return "no reliable difference"
        a_lower = r["mean_diff"] <0
        return r["A"] if a_lower == LOWER_IS_BETTER[r["metric"]] else r["B"]

    df["better_side"] = df.apply(winner,axis=1)
    return df

In [4]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
rand = np.random.default_rng(SEED)
long = load_long()

# Pivot so each row is an image and each column is a (metric, model, method) combination, making it easy to comparemethods or models image by image
piv = long.pivot(index="image_index",columns=["metric","model","method"],values="value")

# 1. Mean and 95% bootstrap CI for every metric/model/method
rows=[]
for metric in LOWER_IS_BETTER:
    for model in MODELS:
        for method in METHODS+["Random"]:
            if (metric, model, method) not in piv.columns:
                continue
            m, low, high, n = boot_ci(piv[(metric,model,method)].values,rand)
            rows.append(dict(metric=metric,model=model,method=method,n=n,mean=m,ci_low=low,ci_high=high))

ci=pd.DataFrame(rows)
ci.to_csv(OUTPUT_DIR/"ci_means.csv",index=False)

# Formatted [low,high] string for a compact report table
ci["mean [95% CI]"] = ci.apply(lambda r: f"{r['mean']:.4f} [{r['ci_low']:.4f}, {r['ci_high']:.4f}]", axis=1)
ci.pivot(index=["model","method"], columns="metric",values="mean [95% CI]") \
      .to_csv(OUTPUT_DIR / "table_mean_ci.csv")

# 2. Method vs method 
# Tests every pair of XAI methods 
rows=[]
for metric in LOWER_IS_BETTER:
    for model in MODELS:
        for A,B in combinations(METHODS,2):
            rows.append(dict(metric=metric,model=model,A=A,B=B,
                             **paired_test(piv[(metric,model,A)],piv[(metric,model,B)],rand)))

# Holm correction is applied seperately within each metric
pm = add_holm_and_winner(pd.DataFrame(rows),family_col="metric")
pm.to_csv(OUTPUT_DIR/"paired_methods.csv",index=False)

# 3. Architecture comparison (EfficientNetB0 vs ResNet50, for each method)
rows=[]
for metric in LOWER_IS_BETTER:
    for method in METHODS:
        rows.append(dict(metric=metric,method=method,A="efficientnetb0",B="resnet50",
                         **paired_test(piv[(metric,"efficientnetb0",method)],
                                       piv[(metric,"resnet50",method)],rand)))
pa=add_holm_and_winner(pd.DataFrame(rows),family_col="metric")
pa.to_csv(OUTPUT_DIR/"paired_architectures.csv",index=False)

# 4. Each method vs Random Baseline (deletion/insertion only)
rows=[]
for metric in ["irof_auc","del_auc","ins_auc"]:
    for model in MODELS:
        for method in METHODS:
            rows.append(dict(metric=metric,model=model,A=method,B="Random",
                             **paired_test(piv[(metric,model,method)],
                                           piv[(metric,model,"Random")],
                                           rand)))
pr= add_holm_and_winner(pd.DataFrame(rows),family_col="metric")
pr.to_csv(OUTPUT_DIR/"vs_random.csv",index=False)

In [5]:
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)


def show(df: pd.DataFrame) -> pd.DataFrame:
    """Filters, rounds, and resets the index of the comparison DataFrame."""
    optional_cols = [col for col in ("model", "method") if col in df.columns]

    core_cols = [
        "A",
        "B",
        "n_pairs",
        "mean_diff",
        "ci_low",
        "ci_high",
        "p_holm",
        "evidence",
        "better_side",
    ]

    keep_columns = ["metric"] + optional_cols + core_cols
    return df[keep_columns].round(4).reset_index(drop=True)


display(Markdown("### Mean [95% bootstrap CI]"))
pivot_df = ci.pivot(
    index=["model", "method"], columns="metric", values="mean [95% CI]"
)
display(pivot_df)

display(Markdown("### Method vs method (diff = A − B, paired by image)"))
display(show(pm))

display(
    Markdown(
        "### Architecture (diff = EfficientNetB0 − ResNet50, paired by image)"
    )
)
display(show(pa))

display(Markdown("### Method vs Random baseline (diff = method − Random)"))
display(show(pr))

print(f"Saved CSVs to {OUTPUT_DIR}")


### Mean [95% bootstrap CI]

metric                                  del_auc                  ins_auc                 irof_auc          max_sensitivity                ssim_dist
model          method                                                                                                                              
efficientnetb0 GradCAM  0.1975 [0.1632, 0.2360]  0.4292 [0.3768, 0.4825]  0.3206 [0.2682, 0.3776]  1.0284 [0.9748, 1.0840]  0.7347 [0.7123, 0.7566]
               LIME     0.2279 [0.1969, 0.2602]  0.3692 [0.3263, 0.4130]  0.4030 [0.3542, 0.4557]  1.5800 [1.5363, 1.6259]  0.9901 [0.9861, 0.9942]
               Random   0.1477 [0.1196, 0.1798]  0.3124 [0.2736, 0.3528]  0.4734 [0.4151, 0.5379]                      NaN                      NaN
               SHAP     0.1853 [0.1552, 0.2175]  0.4400 [0.3824, 0.4981]  0.3100 [0.2606, 0.3646]  1.3514 [1.2963, 1.4065]  0.8848 [0.8600, 0.9093]
resnet50       GradCAM  0.1996 [0.1702, 0.2304]  0.4469 [0.4071, 0.4880]  0.3946 [0.3386, 0.4557]  0.8591 [0.8057, 0.9132]  0.5854 [0.5551, 0.6144]
               LIME     0.1902 [0.1556, 0.2267]  0.3042 [0.2700, 0.3395]  0.4490 [0.3682, 0.5359]  1.4243 [1.3640, 1.4884]  0.9622 [0.9573, 0.9671]
               Random   0.1774 [0.1205, 0.2409]  0.2867 [0.2407, 0.3344]  0.4186 [0.3397, 0.5031]                      NaN                      NaN
               SHAP     0.1696 [0.1402, 0.2018]  0.4756 [0.4302, 0.5196]  0.3315 [0.2574, 0.4110]  1.1010 [1.0426, 1.1616]  0.7393 [0.7151, 0.7641]

### Method vs method (diff = A − B, paired by image)

,metric,model,A,B,n_pairs,mean_diff,ci_low,ci_high,p_holm,evidence,better_side
0,irof_auc,resnet50,GradCAM,LIME,81,-0.0544,-0.1021,-0.0098,0.8066,mixed,no reliable difference
1,irof_auc,resnet50,GradCAM,SHAP,81,0.0631,0.0295,0.0955,0.0000,both,SHAP
2,irof_auc,resnet50,LIME,SHAP,81,0.1175,0.0712,0.1650,0.0000,both,SHAP
3,irof_auc,efficientnetb0,GradCAM,LIME,81,-0.0824,-0.1200,-0.0439,0.0000,both,GradCAM
4,irof_auc,efficientnetb0,GradCAM,SHAP,81,0.0105,-0.0253,0.0473,0.8066,none,no reliable difference
5,irof_auc,efficientnetb0,LIME,SHAP,81,0.0930,0.0591,0.1251,0.0000,both,SHAP
6,del_auc,resnet50,GradCAM,LIME,81,0.0094,-0.0204,0.0397,0.8460,none,no reliable difference
7,del_auc,resnet50,GradCAM,SHAP,81,0.0300,0.0077,0.0528,0.0085,both,SHAP
8,del_auc,resnet50,LIME,SHAP,81,0.0206,-0.0146,0.0555,0.8460,none,no reliable difference
9,del_auc,efficientnetb0,GradCAM,LIME,81,-0.0304,-0.0658,0.0075,0.0289,mixed,GradCAM


### Architecture (diff = EfficientNetB0 − ResNet50, paired by image)

,metric,method,A,B,n_pairs,mean_diff,ci_low,ci_high,p_holm,evidence,better_side
0,irof_auc,GradCAM,efficientnetb0,resnet50,81,-0.0740,-0.1503,0.0006,0.0526,none,no reliable difference
1,irof_auc,LIME,efficientnetb0,resnet50,81,-0.0460,-0.1390,0.0440,0.7117,none,no reliable difference
2,irof_auc,SHAP,efficientnetb0,resnet50,81,-0.0215,-0.1101,0.0625,0.4802,none,no reliable difference
3,del_auc,GradCAM,efficientnetb0,resnet50,81,-0.0021,-0.0424,0.0394,0.8340,none,no reliable difference
4,del_auc,LIME,efficientnetb0,resnet50,81,0.0377,-0.0029,0.0790,0.1309,none,no reliable difference
5,del_auc,SHAP,efficientnetb0,resnet50,81,0.0158,-0.0195,0.0510,0.4023,none,no reliable difference
6,ins_auc,GradCAM,efficientnetb0,resnet50,81,-0.0177,-0.0800,0.0459,0.9397,none,no reliable difference
7,ins_auc,LIME,efficientnetb0,resnet50,81,0.0650,0.0167,0.1108,0.0267,both,efficientnetb0
8,ins_auc,SHAP,efficientnetb0,resnet50,81,-0.0356,-0.1018,0.0320,0.9397,none,no reliable difference
9,max_sensitivity,GradCAM,efficientnetb0,resnet50,80,0.1685,0.0935,0.2436,0.0001,both,resnet50


### Method vs Random baseline (diff = method − Random)

,metric,model,A,B,n_pairs,mean_diff,ci_low,ci_high,p_holm,evidence,better_side
0,irof_auc,resnet50,GradCAM,Random,81,-0.0239,-0.0796,0.0309,0.8895,none,no reliable difference
1,irof_auc,resnet50,LIME,Random,81,0.0304,-0.0262,0.0860,0.1495,none,no reliable difference
2,irof_auc,resnet50,SHAP,Random,81,-0.0870,-0.1420,-0.0306,0.0001,both,SHAP
3,irof_auc,efficientnetb0,GradCAM,Random,81,-0.1528,-0.2005,-0.1047,0.0000,both,GradCAM
4,irof_auc,efficientnetb0,LIME,Random,81,-0.0704,-0.1284,-0.0153,0.0676,mixed,no reliable difference
5,irof_auc,efficientnetb0,SHAP,Random,81,-0.1634,-0.2098,-0.1193,0.0000,both,SHAP
6,del_auc,resnet50,GradCAM,Random,81,0.0222,-0.0290,0.0720,0.0047,mixed,Random
7,del_auc,resnet50,LIME,Random,81,0.0128,-0.0353,0.0595,0.1557,none,no reliable difference
8,del_auc,resnet50,SHAP,Random,81,-0.0079,-0.0602,0.0417,0.1557,none,no reliable difference
9,del_auc,efficientnetb0,GradCAM,Random,81,0.0498,0.0207,0.0772,0.0001,both,Random


Saved CSVs to /data/aveera/results/ci_results


In [6]:
vr = pr.copy()
vr["outcome"] = np.where(vr["better_side"] == "no reliable difference", "not distinguishable from random",
                 np.where(vr["better_side"] == "Random", "WORSE than random", "better than random"))
vr = vr[["metric","model","A","mean_diff","ci_low","ci_high","p_holm","evidence","outcome"]].round(4)
vr.to_csv(OUTPUT_DIR/"vs_random_outcomes.csv", index=False)
display(vr)

,metric,model,A,mean_diff,ci_low,ci_high,p_holm,evidence,outcome
0,irof_auc,resnet50,GradCAM,-0.0239,-0.0796,0.0309,0.8895,none,not distinguishable from random
1,irof_auc,resnet50,LIME,0.0304,-0.0262,0.0860,0.1495,none,not distinguishable from random
2,irof_auc,resnet50,SHAP,-0.0870,-0.1420,-0.0306,0.0001,both,better than random
3,irof_auc,efficientnetb0,GradCAM,-0.1528,-0.2005,-0.1047,0.0000,both,better than random
4,irof_auc,efficientnetb0,LIME,-0.0704,-0.1284,-0.0153,0.0676,mixed,not distinguishable from random
5,irof_auc,efficientnetb0,SHAP,-0.1634,-0.2098,-0.1193,0.0000,both,better than random
6,del_auc,resnet50,GradCAM,0.0222,-0.0290,0.0720,0.0047,mixed,WORSE than random
7,del_auc,resnet50,LIME,0.0128,-0.0353,0.0595,0.1557,none,not distinguishable from random
8,del_auc,resnet50,SHAP,-0.0079,-0.0602,0.0417,0.1557,none,not distinguishable from random
9,del_auc,efficientnetb0,GradCAM,0.0498,0.0207,0.0772,0.0001,both,WORSE than random
